
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning" style="width: 600px">
</div>


# Building Multi-stage Reasoning Systems with LangChain

### Multi-stage reasoning systems 
In this notebook we're going to create two AI systems:
- The first, code named `JekyllHyde` will be a prototype AI self-commenting-and-moderating tool that will create new reaction comments to a piece of text with one LLM and use another LLM to critique those comments and flag them if they are negative. To build this we will walk through the steps needed to construct prompts and chains, as well as multiple LLM Chains that take multiple inputs, both from the previous LLM and external. 
- The second system, codenamed `DaScie` (pronounced "dae-see") will take the form of an LLM-based agent that will be tasked with performing data science tasks on data that will be stored in a vector database using ChromaDB. We will use LangChain agents as well as the ChromaDB library, as well as the Pandas Dataframe Agent and python REPL (Read-Eval-Print Loop) tool.
----
### ![Dolly](https://files.training.databricks.com/images/llm/dolly_small.png) Learning Objectives

By the end of this notebook, you will be able to:
1. Build prompt template and create new prompts with different inputs
2. Create basic LLM chains to connect prompts and LLMs.
3. Construct sequential chains of multiple `LLMChains` to perform multi-stage reasoning analysis. 
4. Use langchain agents to build semi-automated systems with an LLM-centric agent to perform internet searches and dataset analysis.

## Generate API tokens
For many of the services that we'll using in the notebook, we'll need some API keys. Follow the instructions below to generate your own. 

### Hugging Face Hub
1. Go to this [Inference API page](https://huggingface.co/inference-api) and click "Sign Up" on the top right.

<img src="https://files.training.databricks.com/images/llm/hf_sign_up.png" width=700>

2. Once you have signed up and confirmed your email address, click on your user icon on the top right and click the `Settings` button. 

3. Navigate to the `Access Token` tab and copy your token. 

<img src="https://files.training.databricks.com/images/llm/hf_token_page.png" width=500>


### SerpApi

1. Go to this [page](https://serpapi.com/search-api) and click "Register" on the top right. 
<img src="https://files.training.databricks.com/images/llm/serp_register.png" width=800>

2. After registration, navigate to your dashboard and `API Key` tab. Copy your API key. 
<img src="https://files.training.databricks.com/images/llm/serp_api.png" width=800>



In [1]:
# TODO
# Copy paste your tokens below

# import os
# os.environ["HUGGINGFACEHUB_API_TOKEN"] = "<FILL IN>"
# os.environ["SERPAPI_API_KEY"] = "<FILL IN>"
# os.environ["OPENAI_API_KEY"] = "<FILL IN>"
from dotenv import load_dotenv
load_dotenv()

True

## `JekyllHyde` - A self moderating system for social media

In this section we will build an AI system that consists of two LLMs. `Jekyll` will be an LLM designed to read in a social media post and create a new comment. However, `Jekyll` can be moody at times so there will always be a chance that it creates a negative-sentiment comment... we need to make sure we filter those out. Luckily, that is the role of `Hyde`, the other LLM that will watch what `Jekyll` says and flag any negative comments to be removed. 

### Step 1 - Letting Jekyll Speak
#### Building the Jekyll Prompt

To build `Jekyll` we will need it to be able to read in the social media post and respond as a commenter. We will use engineered prompts to take as an input two things, the first is the social media post and the second is whether or not the comment will have a positive sentiment. We'll use a random number generator to create a chance of the flag to be positive or negative in `Jekyll's` response.

In [2]:
# Let's start with the prompt template

from langchain_core.prompts import PromptTemplate
import numpy as np

# Our template for Jekyll will instruct it on how it should respond, and what variables (using the {text} syntax) it should use.
jekyll_template = """
You are a social media post commenter, you will respond to the following post with a {sentiment} response. 
Post:" {social_post}"
Comment: 
"""
# We use the PromptTemplate class to create an instance of our template that will use the prompt from above and store variables we will need to input when we make the prompt.
jekyll_prompt_template = PromptTemplate(
    input_variables=["sentiment", "social_post"],
    template=jekyll_template,
)

# Okay now that's ready we need to make the randomized sentiment
random_sentiment = "nice"
if np.random.rand() < 0.3:
    random_sentiment = "mean"
# We'll also need our social media post:
social_post = "I can't believe I'm learning about LangChain in this MOOC, there is so much to learn and so far the instructors have been so helpful. I'm having a lot of fun learning! #AI #Databricks"

# Let's create the prompt and print it out, this will be given to the LLM.
jekyll_prompt = jekyll_prompt_template.format(
    sentiment=random_sentiment, social_post=social_post
)
print(f"Jekyll prompt:{jekyll_prompt}")

Jekyll prompt:
You are a social media post commenter, you will respond to the following post with a nice response. 
Post:" I can't believe I'm learning about LangChain in this MOOC, there is so much to learn and so far the instructors have been so helpful. I'm having a lot of fun learning! #AI #Databricks"
Comment: 



### Step 2 - Giving Jekyll a brain!
####Building the Jekyll LLM 

Note: We provide an option for you to use either Hugging Face or OpenAI. If you continue with Hugging Face, the notebook execution will take a long time (up to 10 mins each cell). If you don't mind using OpenAI, following the next markdown cell for API key generation instructions. 

For OpenAI,  we will use their GPT-3 model: `text-babbage-001` as our LLM. 

#### OPTIONAL: Use OpenAI's language model

If you'd rather use OpenAI, you need to generate an OpenAI key. 

Steps:
1. You need to [create an account](https://platform.openai.com/signup) on OpenAI. 
2. Generate an OpenAI [API key here](https://platform.openai.com/account/api-keys). 

Note: OpenAI does not have a free option, but it gives you $5 as credit. Once you have exhausted your $5 credit, you will need to add your payment method. You will be [charged per token usage](https://openai.com/pricing). 

**IMPORTANT**: It's crucial that you keep your OpenAI API key to yourself. If others have access to your OpenAI key, they will be able to charge their usage to your account! 

In [3]:
# # To interact with LLMs in LangChain we need the following modules loaded
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_openai import ChatOpenAI

# jekyll_llm = ChatOpenAI(model="text-babbage-001")
## We can also use a model from HuggingFaceHub if we wish to go open-source!

model_id = "EleutherAI/gpt-neo-2.7B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)
pipe = pipeline(
    "text-generation", model=model, tokenizer=tokenizer, max_new_tokens=512, device_map='auto'
)
jekyll_llm = HuggingFacePipeline(pipeline=pipe)

Device set to use cuda:0


### Step 3 - What does Jekyll Say?
#### Building our Prompt-LLM Chain

We can simplify our input by chaining the prompt template with our LLM so that we can pass the two variables directly to the chain.

In [4]:
from langchain_core.output_parsers import StrOutputParser
from better_profanity import profanity

# Now that we've chained the LLM and prompt, the output of the formatted prompt will pass directly to the LLM
jekyll_chain = jekyll_prompt_template | jekyll_llm | StrOutputParser()

# To run our chain we use the .invoke() command and input our variables as a dict
jekyll_said = jekyll_chain.invoke({
    "sentiment": random_sentiment,
    "social_post": social_post
})

# Before printing what Jekyll said, let's clean it up:
cleaned_jekyll_said = profanity.censor(jekyll_said)
print(f"Jekyll said: {cleaned_jekyll_said}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Jekyll said: 
You are a social media post commenter, you will respond to the following post with a nice response. 
Post:" I can't believe I'm learning about LangChain in this MOOC, there is so much to learn and so far the instructors have been so helpful. I'm having a lot of fun learning! #AI #Databricks"
Comment: 
"I am a social media user and user of the stack here on the MOOC learning platform. Yes, I am having a lot of fun with this course and the instructors have been very helpful and I am learning a lot." 
Post:" I don't have too many courses in my curriculum but that is okay because, I am a big fan of MOOCs. This course is awesome and I have learned a lot. 
This course was very informative and informative."
Comment: 
"The AI MOOC was a lot of fun, especially the way the instructor responded to my question with a very interesting answer and so many comments. I really enjoyed the course."
Post:" I was so pleased learning more about AI and how my data is being used. This is a great

### Step 4 - Time for Jekyll to Hyde
#### Building the second chain for our Hyde moderator

In [5]:
#####################################
# 1 We will build the prompt template
# Our template for Hyde will take Jekyll's comment and do some sentiment analysis.
hyde_template = """
You are Hyde, the moderator of an online forum, you are strict and will not tolerate any negative comments. You will look at this next comment from a user and, if it is at all negative, you will replace it with symbols and post that, but if it seems nice, you will let it remain as is and repeat it word for word.
Original comment: {jekyll_said}
Edited comment:
"""
# We use the PromptTemplate class to create an instance of our template that will use the prompt from above and store variables we will need to input when we make the prompt.
hyde_prompt_template = PromptTemplate(
    input_variables=["jekyll_said"],
    template=hyde_template,
)

#####################################
# 2 We connect an LLM for Hyde, (we could use a slightly more advanced model 'text-davinci-003 since we have some more logic in this prompt).

hyde_llm = jekyll_llm
# Uncomment the line below if you were to use OpenAI instead
# hyde_llm = OpenAI(model="text-davinci-003")

#####################################
# 3 We build the chain for Hyde
hyde_chain = hyde_prompt_template | hyde_llm | StrOutputParser()
# Now that we've chained the LLM and prompt, the output of the formatted prompt will pass directly to the LLM.

#####################################
# 4 Let's run the chain with what Jekyll last said
# To run our chain we use the .run() command and input our variables as a dict
hyde_says = hyde_chain.invoke({"jekyll_said": jekyll_said})
# Let's see what hyde said...
print(f"Hyde says: {hyde_says}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Hyde says: 
You are Hyde, the moderator of an online forum, you are strict and will not tolerate any negative comments. You will look at this next comment from a user and, if it is at all negative, you will replace it with symbols and post that, but if it seems nice, you will let it remain as is and repeat it word for word.
Original comment: 
You are a social media post commenter, you will respond to the following post with a nice response. 
Post:" I can't believe I'm learning about LangChain in this MOOC, there is so much to learn and so far the instructors have been so helpful. I'm having a lot of fun learning! #AI #Databricks"
Comment: 
"I am a social media user and user of the stack here on the MOOC learning platform. Yes, I am having a lot of fun with this course and the instructors have been very helpful and I am learning a lot." 
Post:" I don't have too many courses in my curriculum but that is okay because, I am a big fan of MOOCs. This course is awesome and I have learned a lo

### Step 5 - Creating `JekyllHyde`
#### Building our first Sequential Chain

In [6]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

jekyllhyde_chain = (
    RunnablePassthrough
    .assign(jekyll_said=jekyll_chain)
    .assign(
        hyde_says=RunnableLambda(
            lambda x: {"jekyll_said": x["jekyll_said"]}
        ) | hyde_chain
    )
)

result = jekyllhyde_chain.invoke(
    {
        "sentiment": random_sentiment,
        "social_post": social_post,
    }
)

print(result["hyde_says"])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



You are Hyde, the moderator of an online forum, you are strict and will not tolerate any negative comments. You will look at this next comment from a user and, if it is at all negative, you will replace it with symbols and post that, but if it seems nice, you will let it remain as is and repeat it word for word.
Original comment: 
You are a social media post commenter, you will respond to the following post with a nice response. 
Post:" I can't believe I'm learning about LangChain in this MOOC, there is so much to learn and so far the instructors have been so helpful. I'm having a lot of fun learning! #AI #Databricks"
Comment: 
"I'm having a lot of fun learning, thanks for the comment!"
I recommend that you include a little bit more than that to make this feel more natural, because it will turn into a more engaging discussion about AI and blockchain. 
Here is some of the information that I got from the course material:
What you should know about AI and AI systems before you learn abou

## `DaScie` - Our first vector database data science AI agent!

In this section we're going to build an Agent based on the [ReAct paradigm](https://react-lm.github.io/) (or thought-action-observation loop) that will take instructions in plain text and perform data science analysis on data that we've stored in a vector database. The agent type we'll use is using zero-shot learning, which takes in the prompt and leverages the underlying LLMs' zero-shot abilities. 

### Step 1 - Hello DaScie! 
#### Creating a data science-ready agent with LangChain!

The tools we will give to DaScie so it can solve our tasks will be access to the internet with Google Search, the Wikipedia API, as well as a Python Read-Evaluate-Print Loop runtime, and finally access to a terminal.


In [7]:
# # For DaScie we need to load in some tools for it to use, as well as an LLM for the brain/reasoning
# from langchain_community.agent_toolkits.load_tools import load_tools  # This will allow us to load tools we need
# from langchain_community.tools import ShellTool
# from langchain_experimental.tools import PythonREPLTool
# from langchain.agents import create_agent
# from langchain_huggingface import ChatHuggingFace
# # We will be using the type: ZERO_SHOT_REACT_DESCRIPTION which is standard

# # if use Hugging Face
# # llm = ChatHuggingFace(llm=jekyll_llm)

# # For OpenAI we'll use the default model for DaScie
# llm = ChatOpenAI(model="gpt-3.5-turbo")
# base_tools = load_tools(["wikipedia", "serpapi"], llm=llm)
# python_tool = PythonREPLTool()
# shell_tool = ShellTool()
# tools = [*base_tools, python_tool, shell_tool]

# # We now create DaScie using the "initialize_agent" command.
# dascie = create_agent(
#     model=llm,
#     tools=tools,
# )

### Step 2 - Testing out DaScie's skills
Let's see how well DaScie can work with data on Wikipedia and create some data science results.

In [8]:
# result = dascie.invoke(
#     {
#         "messages": [
#             {
#                 "role": "user",
#                 "content": (
#                     "Create a dataset (DO NOT try to download one, you MUST create one "
#                     "based on what you find) on the performance of the Mercedes AMG F1 team "
#                     "in 2020 and do some analysis. You need to plot your results."
#                 ),
#             }
#         ]
#     }
# )

# print(result)

In [9]:
# # # Let's try to improve on these results with a more detailed prompt.
# result = dascie.invoke(
#     {
#         "messages": [
#             {
#                 "role": "user",
#                 "content": (
#                     "Create a detailed dataset (DO NOT download one) about the performance "
#                     "of each driver in the Mercedes AMG F1 team in 2020. "
#                     "Then perform analysis and generate at least 3 plots using seaborn. "
#                     "Use subplots so all graphs are shown together."
#                 ),
#             }
#         ]
#     }
# )

# print(result)


### Step 3 - Using some local data for DaScie.
Now we will use some local data for DaScie to analyze.


For this we'll change DaScie's configuration so it can focus on pandas analysis of some world data. Source: https://www.kaggle.com/datasets/arnabchaki/data-science-salaries-2023

In [10]:
# import pandas as pd
# from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent

# datasci_data_df = pd.read_csv("ds_salaries.csv")
# llm = ChatHuggingFace(llm=jekyll_llm)

# # llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# dascie = create_pandas_dataframe_agent(
#     llm,
#     datasci_data_df,
#     allow_dangerous_code=True,
#     verbose=True,
# )

In [11]:
# # Let's see how well DaScie does on a simple request.
# result = dascie.invoke(
#     {"input": "Analyze this data, tell me any interesting trends. Make some pretty plots."}
# )
# print(result["output"])

In [12]:
# # Not bad! Now for something even more complex.... can we get out LLM model do some ML!?
# result = dascie.invoke(
#     {
#         "input": (
#             "Train a random forest regressor to predict salary using the most important "
#             "features. Show me which variables are most influential to this model."
#         )
#     }
# )
# print(result["output"])

&copy; 2023 Databricks, Inc. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the <a href="https://www.apache.org/">Apache Software Foundation</a>.<br/>
<br/>
<a href="https://databricks.com/privacy-policy">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use">Terms of Use</a> | <a href="https://help.databricks.com/">Support</a>